Connected to Python 3.13.3

In [ ]:
"""
Online stable matching: babies and daycares. Final Assignment: Algorithmic Decision Making
Can & Femke

match[b] = daycare number of baby b, or -1 if baby b has no place.
Babies arrive in the order 0, 1, 2, ...
"""
import math
import random
import csv
import os

# ---------------------------------------------------------------
# Step 1: the model
# ---------------------------------------------------------------
def make_instance(n, q_min, q_max, L, random_prefs=True):
    """
    n = number of daycares
    q = capacity of every daycare
    L = load (babies / total spots available)
    """
    caps = [random.randint(q_min, q_max) for _ in range(n)]
    Q = sum(caps)
    m = math.ceil(L * Q)

    # random spots in the 1x1 square, each spot is (x, y)
    daycare_pos = [(random.random(), random.random()) for _ in range(n)]
    baby_pos = [(random.random(), random.random()) for _ in range(m)]

    # dist[b][d] = distance from baby b to daycare d, divided by sqrt(2)
    # so that every distance is between 0 and 1
    dist = []

    for b in range(m):
        row = []
        for d in range(n):
            dx = baby_pos[b][0] - daycare_pos[d][0]
            dy = baby_pos[b][1] - daycare_pos[d][1]
            row.append(math.sqrt(dx ** 2 + dy ** 2) / math.sqrt(2))
        dist.append(row)

    baby_order = []  # baby_order[b] = daycares of baby b, favorite first
    baby_rank = []  # baby_rank[b][d] = 1 if d is the favorite of b, 2 if second, ...

    for b in range(m):
        if random_prefs:
            order = list(range(n))  # [0, 1, 2, ..., n-1]
            random.shuffle(order)  # put them in a random order
        else:
            order = sorted(range(n), key=lambda d: dist[b][d])  # we could use this in case the babies preferences are based on distance
        rank = [0] * n

        for position, d in enumerate(order):
            rank[d] = position + 1
        baby_order.append(order)  # lists baby b's daycares from favorite to least favorite
        baby_rank.append(rank)  # gives position of daycare d in baby b's ranking (1 is best)

    return {"n": n, "caps": caps, "m": m, "dist": dist,
            "baby_order": baby_order, "baby_rank": baby_rank}


# ---------------------------------------------------------------
# Step 2: the measures
# ---------------------------------------------------------------
def evaluate(inst, match):
    n, caps, m = inst["n"], inst["caps"], inst["m"]
    dist, baby_rank = inst["dist"], inst["baby_rank"]

    # 1. matching size and 2. average rank, 3. average distance (how close assigned babies live to their daycares)
    size = 0
    rank_sum = 0
    distance_sum = 0

    for b in range(m):
        if match[b] != -1:
            size += 1
            rank_sum += baby_rank[b][match[b]]
            distance_sum += dist[b][match[b]]

    avg_rank = rank_sum / size if size > 0 else float("nan")
    avg_distance = distance_sum / size if size > 0 else float("nan")

    # Fraction of all daycare places that are filled
    utilisation = size / sum(caps)

    # For each daycare: how many babies it has, and its farthest baby
    load = [0] * n
    worst = [-1] * n  # distance of its farthest baby

    for b in range(m):
        d = match[b]
        if d != -1:
            load[d] += 1
            worst[d] = max(worst[d], dist[b][d])

    # blocking pairs: baby b and daycare d block if
    #    - b likes d more than her own match (or has no match), AND
    #    - d has a free place, or d likes b more than its farthest baby
    blocking = 0

    for b in range(m):
        if match[b] == -1:
            my_rank = n + 1  # unmatched: worse than any daycare
        else:
            my_rank = baby_rank[b][match[b]]
        for d in range(n):
            if baby_rank[b][d] < my_rank:
                if load[d] < caps[d] or dist[b][d] < worst[d]:
                    blocking += 1

    return size, avg_rank, blocking, avg_distance, utilisation


# ---------------------------------------------------------------
# Step 3a: baseline, baby-oriented greedy
# ---------------------------------------------------------------
def online_greedy(inst):
    """Every baby takes her favorite daycare that still has a free place."""
    n = inst["n"]
    caps = inst["caps"]
    m = inst["m"]

    load = [0] * n  # load[d] records how many babies are currently assigned to daycare d
    match = [-1] * m  # initially all babies are unmachted and thus equal to -1

    for b in range(m):  # babies arrive one by one
        for d in inst["baby_order"][b]:  # favorite first
            if load[d] < caps[d]:
                match[b] = d
                load[d] += 1
                break
    return match

# ---------------------------------------------------------------
# Step 3b: baseline, daycare-oriented greedy
# ---------------------------------------------------------------
def online_daycare_greedy(inst):
    """Every baby goes to the closest daycare that still has space."""
    n = inst["n"]
    caps = inst["caps"]
    m = inst["m"]
    dist = inst["dist"]

    load = [0] * n
    match = [-1] * m

    for b in range(m):
        best_daycare = -1
        best_distance = float("inf")

        # find the closest daycare with a free place
        for d in range(n):
            if load[d] < caps[d]:
                if dist[b][d] < best_distance:
                    best_daycare = d
                    best_distance = dist[b][d]

        # assign the baby if a daycare has space
        if best_daycare != -1:
            match[b] = best_daycare
            load[best_daycare] += 1

    return match


# ---------------------------------------------------------------
# Step 4: greedy baby + distance threshold
# ---------------------------------------------------------------
def online_threshold(inst, thresh):
    """Same as greedy but a baby can only be matched to its preferred daycare if it is within the right distance. """
    n = inst["n"]
    caps = inst["caps"]
    m = inst["m"]
    dist = inst["dist"]

    load = [0] * n
    match = [-1] * m

    for b in range(m):  # babies arrive one by one
        for d in inst["baby_order"][b]:  # favorite first
            if load[d] < caps[d] and dist[b][d] < thresh:  # add dist threshold condition
                match[b] = d
                load[d] += 1
                break
    return match

# ---------------------------------------------------------------
# Step 5: greedy baby + shrinking distance threshold
# ---------------------------------------------------------------
def online_shrinking_capacity(inst, thresh, power=1.0):
    """
    Same idea as online_threshold, but a daycare's threshold shrinks as it fills up.
    power = 1   -> threshold shrinks linearly with remaining capacity
    """
    n = inst["n"]
    caps = inst["caps"]
    m = inst["m"]
    dist = inst["dist"]

    load = [0] * n
    match = [-1] * m

    for b in range(m):                       # babies arrive one by one
        for d in inst["baby_order"][b]:      # favorite first
            if load[d] < caps[d]:
                remaining_frac = (caps[d] - load[d]) / caps[d]
                eff_threshold = thresh * (remaining_frac ** power)
                if dist[b][d] < eff_threshold:
                    match[b] = d
                    load[d] += 1
                    break
    return match    

# ---------------------------------------------------------------
# Experiment: run many random worlds and take the average
# ---------------------------------------------------------------
def run_experiment(algorithms, n, q_min, q_max, L, runs=100, random_prefs=True):
    totals = {name: [0, 0, 0, 0, 0] for name in algorithms}

    for _ in range(runs):
        inst = make_instance(n, q_min, q_max, L, random_prefs=random_prefs)

        for name, algo in algorithms.items():
            size, avg_rank, blocking, avg_distance, utilisation = evaluate(inst, algo(inst))

            totals[name][0] += size / inst["m"]
            totals[name][1] += avg_rank
            totals[name][2] += blocking
            totals[name][3] += avg_distance
            totals[name][4] += utilisation

    rows = []
    for name in algorithms:
        rate, avg_rank, blocking, avg_distance, utilisation = [
            t / runs for t in totals[name]
        ]

        print(
            f"  {name:28s} "
            f"matched={rate:6.1%}  "
            f"avg_rank={avg_rank:5.2f}  "
            f"blocking_pairs={blocking:9.1f}  "
            f"avg_distance={avg_distance:5.3f}  "
            f"filled={utilisation:6.1%}"
        )

        rows.append({
            "algorithm": name,
            "n": n,
            "q_min": q_min,
            "q_max": q_max,
            "L": L,
            "runs": runs,
            "matched_rate": rate,
            "avg_rank": avg_rank,
            "blocking_pairs": blocking,
            "avg_distance": avg_distance,
            "utilisation": utilisation,
        })

    # record data into .csv
    if csv_path is not None:
        write_header = not os.path.exists(csv_path)
        with open(csv_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            if write_header:
                writer.writeheader()
            writer.writerows(rows)

if __name__ == "__main__":
    random.seed(333)  # for experiment reproducibility

    n = 100           # number of daycares
    q_min = 1         # minimum daycare capacity
    q_max = 25        # maximum daycare capacity
    runs = 500       # 1000 for the final experiments

    algorithms = {
         "Baby-oriented Greedy": online_greedy,
         "Daycare-oriented Greedy": online_daycare_greedy
    }

    # we evaluate distance thresholds 
    thresholds = [round(i / 20, 2) for i in range(1, 21)] # range of 0.05 to 1.00 in intervals of 0.05.

    for threshold in thresholds:
        algorithms[f"Online Threshold {threshold:.2f}"] = (
            lambda inst, t=threshold: online_threshold(inst, t)
        )
        algorithms[f"Shrinking Threshold {threshold:.2f}"] = (
            lambda inst, t=threshold: online_shrinking_capacity(inst, t, power = 1)
        )

    # low demand, balanced demand, and excess demand
    loads = [round(i / 10, 2) for i in range(5, 16)] # range of 0.5 to 1.5 in 0.1 intervals
    #loads = [round(i / 10, 2) for i in range(5, 16)] # range of 1.0 to 1.2 in 0.05 intervals

    csv_path = "results_q1q25_rand_basic+shrink_1000.csv"

    if os.path.exists(csv_path):
        os.remove(csv_path)  # start each run with a clean file, not stale appended rows

    # run experiment for each load
    for L in loads:
        print(
            f"\nn={n}, q_min={q_min}, q_max={q_max}, "
            f"L={L:.2f}, runs={runs}"
        )

        run_experiment(
            algorithms,
            n=n,
            q_min=q_min,
            q_max=q_max,
            L=L,
            runs=runs,
            random_prefs=True,
        )


n=100, q_min=1, q_max=25, L=0.50, runs=500
  Baby-oriented Greedy         matched=100.0%  avg_rank= 1.18  blocking_pairs=     84.1  avg_distance=0.368  filled= 50.0%
  Daycare-oriented Greedy      matched=100.0%  avg_rank=50.52  blocking_pairs=  21602.6  avg_distance=0.043  filled= 50.0%
  Online Threshold 0.05        matched= 71.6%  avg_rank=39.27  blocking_pairs=  28484.0  avg_distance=0.033  filled= 35.8%
  Shrinking Threshold 0.05     matched= 58.3%  avg_rank=41.54  blocking_pairs=  38750.1  avg_distance=0.028  filled= 29.2%
  Online Threshold 0.10        matched= 97.3%  avg_rank=20.87  blocking_pairs=   9921.8  avg_distance=0.066  filled= 48.7%
  Shrinking Threshold 0.10     matched= 87.5%  avg_rank=27.78  blocking_pairs=  20456.5  avg_distance=0.052  filled= 43.7%
  Online Threshold 0.15        matched= 99.8%  avg_rank=11.02  blocking_pairs=   4567.3  avg_distance=0.097  filled= 49.9%
  Shrinking Threshold 0.15     matched= 96.2%  avg_rank=17.99  blocking_pairs=  11301.6  avg_di